# Preparing Arbitrary Quantum States
## Part II. Alias sampling

As a reminder, this kata consists of two parts:

1. [Part I](./ArbitraryStatePreparation_1_Rotations.ipynb) covers rotation-based state preparation algorithms, starting with naive state preparation.
2. Part II (this notebooks) covers alias sampling, an alternative state preparation algorithm that doesn't use arbitrary rotations.

Alias sampling algorithm for state preparation was proposed by Craig Gidney; you can read about it in his blog post ["Roulette Selection fit for a Quantum Computer"](https://algassert.com/post/1805). In this part of this kata, we'll first learn the classical alias sampling algorithm and then see how to apply its core principles to quantum state preparation.

## Classical alias sampling algorithm

The classical alias sampling algorithm solves the roulette selection problem: Given a list of values, pick a random element from the list with probability proportional to its value.

For example, consider the list of values `[7, 1, 3, 5]`. The sum of the values in the list is $16$, and the probabilities of selecting each element in the list should be $\frac7{16}, \frac1{16}, \frac3{16}$, and $\frac5{16}$, respectively.

Alias sampling algorithm reformulates this problem as sampling from a collection of equally likely sub-problems, where each sub-problem involves choosing between only two items. This algorithm turns out to be more efficient (in terms of sampling time) than other roulette selection algorithms. (See the [original blog post](https://algassert.com/post/1805) for comparison.)

Let's illustrate the workings of this algorithm for the example we used above, `[7, 1, 3, 5]`. We can draw the probabilities of picking each value as columns, with each box of each column representing probability $\frac1{16}$. (There are $16$ boxes in the histogram, so the total probability of picking any value is $1$, as expected.)

<center><img src="./images/initial.svg" width=30%/></center>

Next, we rearrange the boxes so that each column is composed of boxes of one or two colors and has height $4$. Here are two examples of how we can do this:

<center>
<table>
  <tr>
    <td style="border: none; padding: 0px; padding-right: 50px;"><img src="./images/repack1.svg" width=100%/></td>
    <td style="border: none; padding: 0px; padding-left: 50px;"><img src="./images/repack2.svg" width=100%/></td>
  </tr>
</table>
</center>

Now, picking a random value looks as follows:

1. Pick a random column (each column has the same probability).
2. Pick a color within that column (the colors and probabilities of choosing each one depend on the choice of the column).
3. Return the value associated with that color.

Before we see how this algorithm can be applied to quantum state preparation, let's see how to implement it as a purely classical algorithm.

### Problem 1. Discretize probabilities

The first step of implementing classical alias sampling algorithm is discretizing the probabilities to represent them as proper fractions with an integer numerator and a denominator that is a power of $2$. (This is not strictly necessary when we're working with floating-point probabilities, but we'll need this once we switch to fixed-point probabilities in quantum state preparation.)

**Inputs:** 

1. A list of $N$ real non-negative numbers `probs` that add up to $1$.
2. An integer $b$ - the number of bits of precision to use for discretization.

**Goal:** 

Return the discretized probabilities `discrete_probs` as a list of $N$ non-negative integers. The normalized probabilities must satisfy two conditions:

1. The sum of all elements in `discrete_probs` must be $2^b$. In other words, dividing each element by $2^b$ must give a valid probability distribution.
2. Each element of `discrete_probs`, divided by $2^b$, must be within $\frac1{2^b}$ absolute tolerance of the corresponding element of `probs`. In other words, the resulting probability distribution must approximate the original probability distribution.

**Examples:**

* `probs = [0.75, 0.25], b = 2` is discretized as `[3, 1]`: $3+1 = 4 = 2^2$, and if you divide each number by $4$, you'll get exactly the original probabilities, no approximation required.
* `probs = [0.3, 0.3, 0.3, 0.1], b = 2` can be discretized as `[1, 1, 1, 1]`: $1 + 1 + 1 + 1 = 4$, and if you divide each number by $4$, you'll get $|0.3 - \frac14| = 0.05 < \frac14$ and $|0.1 - \frac14| = 0.15 < \frac14$. Alternatively, `[2, 1, 1, 0]` is also a valid answer: $|0.3 - \frac24| = 0.2 < \frac14$ and $|0.1 - \frac04| = 0.1 < \frac14$.

In [ ]:
from test_ArbitraryStatePreparation import problem

@problem
def discretize_probabilities(probs: list[float], b: int) -> list[int]:
    # Write your code here
    ...

### Problem 2. Repack the values

**Inputs:** 

1. An integer $n \ge 1$.
2. An integer $b \ge 0$.
3. A list of $2^n$ non-negative integer values that add up to $2^{n + b}$. Each element of the list, when divided by $2^{n + b}$, is the probability of getting the corresponding outcome during sampling.

**Goal:** "Repack" those values into $2^n$ columns of height $2^b$ described as a list of $2^n$ two-element tuples as follows:

* Each tuple consists of two integers, "alias" and "threshold".
* Tuple with index $i$ describes column $i$ of the repacking. This column consists of two colors: $threshold$ boxes of color $i$ and $2^b - threshold$ boxes of color $alias$. The $alias$ value is allowed to be equal to the column index $i$.

**Example:**

For the example we saw earlier in the kata, the inputs are $n = 2$, $b = 2$, and `values = [7, 3, 1, 5]`. 
Here is how the first repacking of these values is represented as a list of alias-threshold tuples in the format described above:

<center>
<table>
  <tr>
    <td style="border: none; padding: 0px; padding-right: 50px;"><img src="./images/repack1.svg" width=100%/></td>
    <td style="border: none; padding: 0px;">
        <pre>[(0, 4), (0, 3), (3, 1), (0, 2)]</pre>
        or<br/>
        <pre>[(0, 0), (0, 3), (3, 1), (0, 2)]</pre>
    </td>
  </tr>
</table>
</center>

The first column consists of $4$ boxes of the same color. This can be represented either as `threshold = 4` with any value of alias ($4$ boxes of color $0$ and $0$ boxes of color `alias`) or as `alias = 0, threshold = 0` ($0$ boxes of color $0$ and $4$ boxes of color `alias` which is also $0$).

The rest of the columns consist of two colors each, so for them the representation is unique: `threshold` is the number of boxes of the color that matches the column index, and `alias` is the color of the boxes on top of the column (the ones that don't match the column index).

Similarly, the second repacking can be represented as follows:
<center>
<table>
  <tr>
    <td style="border: none; padding: 0px; padding-right: 50px;"><img src="./images/repack2.svg" width=100%/></td>
    <td style="border: none; padding: 0px;">
        <pre>[(0, 4), (3, 3), (0, 1), (0, 4)]</pre>
        or<br/>
        <pre>[(0, 0), (3, 3), (0, 1), (3, 0)]</pre>
    </td>
  </tr>
</table>
</center>

This time, both the first and the last columns have ambiguous representations.

In [ ]:
from test_ArbitraryStatePreparation import problem

@problem
def repack(n: int, b: int, values: list[int]) -> list[tuple[int, int]]:
    # Write your code here
    ...

### Problem 3. Classical sampling

**Inputs:** 

1. An integer $b \ge 0$.
2. The list of values "repacked" into $2^n$ columns, each of height $2^b$, in the format that matches the output format of the previous problem.

**Goal:** 
Return a single value sampled from the original probability distribution (the one provided as the input to the first problem). In other words, the probability of returning each value $x$ from $0$ to $2^{n} - 1$, inclusive, should be proportional to the number of boxes of color $x$ in the repacked values.

In [ ]:
from test_ArbitraryStatePreparation import problem

@problem
def sample(b: int, repacked_values: list[tuple[int, int]]) -> int:
    # Write your code here
    ...

## Using alias sampling principles for state preparation

How can we use the principles of this classical sampling algorithm to prepare a quantum state?

$\color{red}{TODO}$

Theory: how to translate classical sampling into quantum state preparation

Choose which column to sample from -> USP to prepare all columns with equal weights
Big second step to figure out which value to choose -> Rearranging the weights between columns


Let's start with a small example of preparing a single-qubit state: $n = 1$, $b = 2$, `probs=[0.25, 0.75]`. 

These probabilities are already discretized with the required precision: $probs = [\frac28, \frac68]$.
Their repacking is `[1, 2], [1, 0]`: the first column of repacking has $2$ boxes of color $0$ and $2$ boxes of color $1$, and the second column has $4$ boxes of color $1$ which is treated as the alias for this column.

Let's follow the steps of the algorithm and see how it evolves over time.

1. Prepare uniform superposition on the `state` register.

   $$\tfrac1{\sqrt2} (\ket{0} + \ket{1})_{state}$$

2. Allocate `alias`, `threshold`, and `coin_toss` registers, alias with $1$ qubit and the other two with $2$ qubits each.

   $$\tfrac1{\sqrt2} (\ket{0} + \ket{1})_{state} \ket{0}_{alias} \ket{0}_{threshold} \ket{0}_{coin\_toss}$$

3. Load aliases and thresholds (classical data) into `alias` and `threshold` registers.  
   For each basis state of the `state` register, the state of the corresponding `alias` and `threshold` basis states will change to hold the corresponding classical value.

   $$\tfrac1{\sqrt2} (\ket{0}_{state} \ket{\color{red}{\textbf{1}}}_{alias} \ket{\color{red}{\textbf{2}}}_{threshold} + 
                      \ket{1}_{state} \ket{\color{red}{\textbf{1}}}_{alias} \ket{\color{red}{\textbf{0}}}_{threshold}) \ket{0}_{coin\_toss}$$

   > Note that we're writing the basis states of multi-qubit registers as a single integer rather than a bitstring; this simplifies working with them as integers.

4. Prepare uniform superposition on the `coin_toss` register.

   $$\ket{0}_{coin\_toss} \rightarrow \tfrac12 (\ket{0} + \ket{1} + \ket{2} + \ket{3})_{coin\_toss}$$

5. Perform the conditional swap: swap `state` register with `alias` register if `coin_toss >= threshold`.

   At this stage, all basis states have the same amplitude, $\frac1{2\sqrt2}$, so we'll omit the normalization coefficient and focus on the transformations of the individual basis states.

   | Before | After |
   |:--------:|:-------:|
   | $\ket{0}_{state} \ket{1}_{alias} \ket{2}_{threshold} \ket{0}_{coin\_toss}$ | $\ket{0}_{state} \ket{1}_{alias} \ket{2}_{threshold} \ket{0}_{coin\_toss}$ |
   | $\ket{0}_{state} \ket{1}_{alias} \ket{2}_{threshold} \ket{1}_{coin\_toss}$ | $\ket{0}_{state} \ket{1}_{alias} \ket{2}_{threshold} \ket{1}_{coin\_toss}$ |
   | $\ket{0}_{state} \ket{1}_{alias} \ket{2}_{threshold} \ket{2}_{coin\_toss}$ | $\ket{\color{red}{\textbf{1}}}_{state} \ket{\color{red}{\textbf{0}}}_{alias} \ket{2}_{threshold} \ket{2}_{coin\_toss}$ |
   | $\ket{0}_{state} \ket{1}_{alias} \ket{2}_{threshold} \ket{3}_{coin\_toss}$ | $\ket{\color{red}{\textbf{1}}}_{state} \ket{\color{red}{\textbf{0}}}_{alias} \ket{2}_{threshold} \ket{3}_{coin\_toss}$ |
   | $\ket{1}_{state} \ket{1}_{alias} \ket{0}_{threshold} \ket{0}_{coin\_toss}$ | $\ket{\color{red}{\textbf{1}}}_{state} \ket{\color{red}{\textbf{1}}}_{alias} \ket{0}_{threshold} \ket{0}_{coin\_toss}$ |
   | $\ket{1}_{state} \ket{1}_{alias} \ket{0}_{threshold} \ket{1}_{coin\_toss}$ | $\ket{\color{red}{\textbf{1}}}_{state} \ket{\color{red}{\textbf{1}}}_{alias} \ket{0}_{threshold} \ket{1}_{coin\_toss}$ |
   | $\ket{1}_{state} \ket{1}_{alias} \ket{0}_{threshold} \ket{2}_{coin\_toss}$ | $\ket{\color{red}{\textbf{1}}}_{state} \ket{\color{red}{\textbf{1}}}_{alias} \ket{0}_{threshold} \ket{2}_{coin\_toss}$ |
   | $\ket{1}_{state} \ket{1}_{alias} \ket{0}_{threshold} \ket{3}_{coin\_toss}$ | $\ket{\color{red}{\textbf{1}}}_{state} \ket{\color{red}{\textbf{1}}}_{alias} \ket{0}_{threshold} \ket{3}_{coin\_toss}$ |

   Notice that for the last four lines ($\ket{1}_{state}$, corresponding to the second column of repacked values) the swap happens for all basis states, but since `alias` matches `state`, this doesn't visibly change them.

At the end of the algorithm, we have a superposition with $2$ terms in which the `state` register is in the $\ket{0}$ state, and $6$ terms in which it is in the $\ket{1}$ state, which gives us our desired probability distribution $[\frac28, \frac68]$.

Now that we've seen the steps of the algorithm in action, let's write some code!

### Problem 4. Load data

**Inputs:** 

1. A `Qubits` register `index` of length $n$ in an arbitrary superposition state.
2. A `Qubits` register `target` of length $m$ in the $\ket{0}$ state.
3. A list of integers `data`, represented as `list[int]`. `data` will have at most $2^n$ elements, each element will be between $0$ and $2^m-1$, inclusive.

**Goal:** Load the data into register `target`, indexed by the value of the register `index`. In other words, for each basis state $\ket{j}$ that is within the range of indices of `data`, transform the input state as follows:

$$\ket{j}_{index}\ket{0}_{target} \rightarrow \ket{j}_{index}\ket{data_j}_{target}$$

For any basis states outside of range of indices of `data`, leave the corresponding components of the input state unchanged.

> Both the index $\ket{j}$ and the value $\ket{data_j}$ should be expressed in binary using little-endian encoding.

In [ ]:
from psiqdk.workbench import Qubits, Qubrick
from test_ArbitraryStatePreparation import problem

@problem
class DataLoader(Qubrick):
    def _compute(self, index: Qubits, target: Qubits, data: list[int]) -> None:
        """Load the data into register `target`, indexed by the value of the register `index`."""
        # Write your code here
        ...

### Problem 5. Swap main register with alias state conditioned on comparison result

**Inputs:** 

1. A `Qubits` register `state_reg` of length $n$ - the register we want to prepare in the superposition state described by the previously given probability distribution.
2. A `Qubits` register `alias` of length $n$ - the register that stores the alias values produced by the repacking algorithm from problem 2.
3. A `Qubits` register `threshold` of length $b$ - the register that stores the threshold values produced by the repacking algorithm from problem 2. These values are stored as fixed-point values between $0$ and $1$ with precision $b$.
4. A `Qubits` register `coin_toss` of length $b$ - the register that stores a fixed-point value between $0$ and $1$ with precision $b$ which we will use to emulate the biased coin toss.

Each register can be in an arbitrary superposition state.

**Goal:** 
Compare the values stored in registers `coin_toss` and `threshold`. If the `coin_toss` value is _greater than or equal to_ `threshold`, swap the states of `state_reg` and `alias`; otherwise, do nothing.

> Feel free to use library Qubricks provided by Workbench for parts of your solution, including data loading!

In [ ]:
from psiqdk.workbench import Qubits, Qubrick
from test_ArbitraryStatePreparation import problem

@problem
class AliasStateSwap(Qubrick):
    def _compute(self, state_reg: Qubits, alias: Qubits, threshold: Qubits, coin_toss: Qubits) -> None:
        """Swap `state_reg` and `alias` if `coin_toss` >= `threshold`."""
        # Write your code here
        ...

### Problem 6. Complete alias sampling state preparation

**Inputs:** 

1. A `Qubits` register `state_reg` of length $n$ - the register we want to prepare in a superposition state.
2. A list of $2^n$ real non-negative numbers `probs` that add up to $1$.
3. An integer $b \ge 1$ - the number of additional bits of precision to use for discretization.

**Goal:**

Prepare the register `state_reg` in a superposition state (possibly entangled with other registers) for which the probability of each basis state $j$ is within $\frac1{2^{n + b}}$ absolute tolerance of the corresponding element of `probs`.

**Example:**

For example, for inputs $n = 2, b = 2$, and `probs=[0.45, 0.2, 0.05, 0.3]`, the probabilities of the expected state are `[0.4375, 0.1875, 0.0625, 0.3125]`, or, in other words, $[\frac7{16}, \frac3{16}, \frac1{16}, \frac5{16}]$. This corresponds to the example `values = [7, 3, 1, 5]` that we used in the discussion of the classical alias sampling algorithm.

In [ ]:
from psiqdk.workbench import Qubits, Qubrick
from test_ArbitraryStatePreparation import problem

@problem
class AliasSamplingStatePrep(Qubrick):
    def _compute(self, state_reg: Qubits, probs: list[float], b: int) -> None:
        """Prepare a state with the given probabilities of each basis state."""
        # Write your code here
        ...

### Demo: Testing alias sampling state preparation routine

Now that we have implemented the end-to-end alias sampling algorithm, let's see how we can test it. 

In part I of this kata, we saw how to use Workbench state vector simulator and its `pull_state` method to test other types of state preparation routines. However, we can not use the exact same approach here. Why? 

Alias sampling leaves the `state` register entangled with other, auxiliary registers: `alias`, `threshold`, and `coin_toss`. This means that we'll either get an unexpected result if we try to pull the state of the `state` register alone or get too much information if we try to use `pull_state` of the entire QPU.

Instead, let's recall that the goal of using alias sampling algorithm was to prepare a superposition state _with the given probabilities of each basis state_. We do not care about the relative phases of each basis state (in fact, we know that none of the basis states are going to have complex or negative amplitudes), we just want to make sure that the probability of each one is correct. And the [Testing and Debugging tutorial](https://docs.construct.psiquantum.com/psiqdk-workbench/new-tutorials/Testing-Debugging.html) points us to another tool that can fetch us exactly this information. The `peek_read_probability` method of a Qubits register returns the probability of the given measurement outcome, in other words, the probability of the given basis state in the overall superposition.

This tells us how to adjust the testing strategy from part I for alias sampling state preparation:

1. Select a set of test cases: the number of qubits in a register $n$, the $2^n$ probabilities describing the target state, and the additional integer parameter $b$.
2. Run the state preparation on each one independently using the state vector simulator.
3. After the preparation routine, loop over all $2^n$ basis states in the `state` register and peek at the read probability of each one. Each probability should be within $\frac1{2^{n+b}}$ absolute tolerance from the expected probability.

The following code shows a demo of testing a modified version of `AliasSamplingStatePrep` Qubrick from problem 6. For simplicity, this Qubrick takes aliases and thresholds as arguments to the compute method instead of computing them on the fly. You can find the full testing logic for problem 6 in the backend file `test_ArbitraryStatePreparation.py`.

> Note that the code goes through just one test case &mdash; the same one we considered at the beginning of the discussion of the quantum algorithm &mdash; and prints the full intermediate quantum states, as well as the resulting quantum circuit. This way, you can check that the state on each step always matches your expectation; this is exactly what you would do when debugging the algorithm implementation!

In [ ]:
from psiqdk.algorithms import SelectNaive
from psiqdk.workbench import QPU, Qubits, Qubrick
from pytest import approx

class AliasSamplingStatePrep(Qubrick):
    def _compute(self, state_reg: Qubits, aliases: list[int], thresholds: list[int], b: int) -> None:
        """Prepare a state with the given probabilities of each basis state."""
        n = len(state_reg)

        print("Step 1: Prepare uniform superposition on the state register")
        state_reg.had()
        self.qc.print_state_vector()

        print("Step 2: Allocate auxiliary registers")
        aliases_reg = self.alloc_temp_qreg(n, "aliases")
        thresholds_reg = self.alloc_temp_qreg(b, "thresholds")
        coin_toss = self.alloc_temp_qreg(b, "coin_toss")
        self.qc.print_state_vector()

        print("Step 3: Load aliases and thresholds into corresponding registers")
        data_loader = SelectNaive()
        data_loader.compute(state_reg, aliases_reg, aliases)
        data_loader.compute(state_reg, thresholds_reg, thresholds)
        self.qc.print_state_vector()

        print("Step 4: Prepare uniform superposition on the coin toss register")
        coin_toss.had()
        self.qc.print_state_vector()

        print("Step 5: Do the swap to finalize the probabilities")
        with coin_toss >= thresholds_reg as cond:
            state_reg.swap(aliases_reg, cond=cond)
        self.qc.print_state_vector()


# Choose test parameters
n = 1
b = 2
probs = [0.25, 0.75]
aliases, thresholds = [1, 1], [2, 0]

# Create QPU (with space for extra qubits) and allocate a fresh register
qpu = QPU(num_qubits=2 * n + 3 * b)
state_reg = Qubits(n, 'state_reg', qpu)
# Instantiate the state prep Qubrick and prepare the state
state_prep = AliasSamplingStatePrep()
state_prep.compute(state_reg, aliases, thresholds, b)
qpu.draw(show_qubricks=True)

# Check that the probability of each basis state matches the expected one
for basis_state in range(1 << n):
    basis_state_prob = state_reg.peek_read_probability(basis_state)
    if basis_state_prob != approx(probs[basis_state], abs = 1 / (1 << (n + b))):
        raise ValueError(f"The probability of basis state {basis_state} is {basis_state_prob}, expected {probs[basis_state]}")

print("Test passed!")

### Discussion: When is alias sampling state preparation useful?

$\color{red}{TODO}$

# Part II conclusion

Congratulations! In the second part of this kata you learned to prepare arbitrary quantum states using alias sampling algorithm.
> Copyright (c) 2026 PsiQuantum